# Dynamic Counting Modeling Choices

This notebook explore several ways on how (not) to resolve the legislative side in reference networks, and analyzed their impacts on a standard metric (centrality).

## 0. Setup

In [1]:
%load_ext autoreload
%autoreload 2

In [33]:
# Imports

from pathlib import Path

import pandas as pd
import matplotlib.pyplot as plt
from itertools import cycle

from network_pluralism.access.resolved import to_level
from network_pluralism.access.utils.files import load_matrix
from network_pluralism.access.utils.labels import (
    get_well_written_sections,
    get_well_written_section,
)
from network_pluralism.access.utils.df import return_top_n
from network_pluralism.setup.paths import COMPUTATION_PATH, PREPROCESSED_DATA_DECISIONS


In [3]:
EXPORT_PATH = Path("../../figures_and_results/dynamic_counting")
EXPORT_PATH.mkdir(parents=True, exist_ok=True)

In [4]:
# Matplotlib Config

def set_rcParams(**kwargs):
    params = {
        "pdf.fonttype": 42,
        "text.usetex": False,
        "font.family": "serif",
        "text.latex.preamble": r"\usepackage{amssymb}\usepackage{amsmath}",
        "font.size": 16,
        "legend.fontsize": 14,
        "legend.title_fontsize": 16,
        "axes.labelsize": 18,
        "xtick.labelsize": 16,
        "ytick.labelsize": 16,
        "axes.spines.right": False,
        "axes.spines.top": False,
        "legend.columnspacing": 1.0,
        "legend.frameon": False,
        "lines.linewidth": 3,
    }
    for param in kwargs:
        if param in plt.rcParams.keys():
            params[param] = kwargs[param]
    for k, v in params.items():
        plt.rcParams[k] = v


set_rcParams()

In [5]:
EXPORT_PATH = Path("../../figures_and_results/dynamic_counting")
EXPORT_PATH.mkdir(exist_ok=True)

In [6]:
matrix, rows_mapping, col_mapping = to_level(0, 1)

Out Matrix of shape (8, 93645)


In [7]:
levels_df = pd.read_parquet(COMPUTATION_PATH / "levels.parquet").set_index("sem_id")[
    ["level"]
]
levels_df.head()

,level
sem_id,
euvtrg_1,1
euvtrg_2,1
euvtrg_3,1
euvtrg_3_1,2
euvtrg_3_2,2


In [8]:
def get_aggregation_matrix_at_level(source_level: int):

    aggregation_matrix, am_rows, am_cols = load_matrix(
        name=f"statutes_level_{source_level}"
    )
    row_labels = list(am_rows.keys())
    col_labels = list(am_cols.keys())

    coo = aggregation_matrix.tocoo()
    edge_list_df = pd.DataFrame(
        {
            "source": [row_labels[i] for i in coo.row],
            "target": [col_labels[j] for j in coo.col],
            "weight": coo.data,
        }
    )
    return edge_list_df

edge_list_df = pd.DataFrame()
for i in range(1,5):
    edge_list_df = pd.concat([edge_list_df,get_aggregation_matrix_at_level(i)])
edge_list_df.drop_duplicates(subset=['source','target'],inplace=True)

### 1. No dynamic Counting

In [9]:
occurrences_df = pd.read_pickle(
    PREPROCESSED_DATA_DECISIONS / "matched_numerical_ids.pkl"
)
occurrences_df.head()

,Unnamed: 0,court,panel,decision,paragraph,pos,semantical_id,start,end,item,level,type,UUID
0,0,BPatG,BPatG_25. Senat,25 W (pat) 508/19,25 W (pat) 508/19_15,0,markeng_8_2_1,292.0,308.0,markeng_8_2_1,3,0,42064170535519776492744549313932832938
1,1,BPatG,BPatG_25. Senat,25 W (pat) 508/19,25 W (pat) 508/19_22,0,markeng_8_2_1,37.0,53.0,markeng_8_2_1,3,0,42064170535519776492744549313932832938
2,3,BPatG,BPatG_25. Senat,25 W (pat) 216/09,25 W (pat) 216/09_3,0,markeng_54_2,96.0,114.0,markeng_54_2,2,0,242215549959440275685037571880796361276
3,4,BPatG,BPatG_25. Senat,25 W (pat) 508/19,25 W (pat) 508/19_21,1,markeng_8_2_1,371.0,387.0,markeng_8_2_1,3,0,42064170535519776492744549313932832938
4,5,BPatG,BPatG_25. Senat,25 W (pat) 508/19,25 W (pat) 508/19_21,0,markeng_64_6,9.0,32.0,markeng_64_6,2,0,317653909489634983075549076632237719954


In [10]:
top_20_raw = (
    occurrences_df.groupby("item")
    .count()[["type"]]
    .rename({"type": "count"}, axis=1)
    .sort_values(by="count", ascending=False)
    .head(20)
    .join(levels_df)
)
top_20_raw

,count,level
item,,
gg_3_1,14742,2
markeng_8_2_1,12053,3
gg_103_1,10122,2
markeng_8_2_2,8755,3
gg_2_1,7913,2
vwgo_132_2_1,6879,3
gg_19_4,6688,2
gg_12_1,6150,2
zpo_97_1,6100,2


In [11]:
top_20_only_level_1 = (
    occurrences_df.query("level==1")
    .groupby("item")
    .count()[["type"]]
    .rename({"type": "count"}, axis=1)
    .sort_values(by="count", ascending=False)
    .head(20)
    .join(levels_df)
)

In [12]:
top_20_only_level_2 = (
    occurrences_df.query("level==2")
    .groupby("item")
    .count()[["type"]]
    .rename({"type": "count"}, axis=1)
    .sort_values(by="count", ascending=False)
    .head(20)
)

In [13]:
top_20_only_level_3 = (
    occurrences_df.query("level==3")
    .groupby("item")
    .count()[["type"]]
    .rename({"type": "count"}, axis=1)
    .sort_values(by="count", ascending=False)
    .head(20)
)
top_20_only_level_3

,count
item,
markeng_8_2_1,12053
markeng_8_2_2,8755
vwgo_132_2_1,6879
vwgo_132_2_3,3801
sgg_160_2_3,3289
fgo_115_2_1,3008
fgo_115_2_2,2897
vwgo_132_2_2,2738
fgo_115_2_3,2694


In [14]:
top_20_only_level_3 = (
    occurrences_df.query("level==4")
    .groupby("item")
    .count()[["type"]]
    .rename({"type": "count"}, axis=1)
    .sort_values(by="count", ascending=False)
    .head(20)
)
top_20_only_level_3

,count
item,
ao_1977_180_1_2_a,565
zpo_551_3_2_b,266
zpo_551_3_2_a,223
ustg_1980_13_1_1_a,182
ao_1977_172_1_2_a,159
fgo_120_3_2_a,146
energiestg_51_1_1_a,130
arbgg_2_1_3_a,74
ao_1977_180_1_2_b,74


In [15]:
top_20_only_level_1_and_2 = occurrences_df.query("level in [1,2]").groupby("item").count()[["type"]].rename(
    {"type": "count"}, axis=1
).sort_values(by="count", ascending=False).head(20).join(levels_df)
top_20_only_level_1_and_2

,count,level
item,,
gg_3_1,14742,2
gg_103_1,10122,2
gg_2_1,7913,2
gg_19_4,6688,2
gg_12_1,6150,2
zpo_97_1,6100,2
stpo_349_2,6029,2
gg_20_3,5356,2
vwgo_154_2,5212,2


In [16]:
top_20_only_level_2_and_3 = occurrences_df.query("level in [3,2]").groupby("item").count()[["type"]].rename(
    {"type": "count"}, axis=1
).sort_values(by="count", ascending=False).head(20).join(levels_df)
top_20_only_level_2_and_3

,count,level
item,,
gg_3_1,14742,2
markeng_8_2_1,12053,3
gg_103_1,10122,2
markeng_8_2_2,8755,3
gg_2_1,7913,2
vwgo_132_2_1,6879,3
gg_19_4,6688,2
gg_12_1,6150,2
zpo_97_1,6100,2


## 2. With dynamic counting

In [17]:
matrix, rows_mapping, col_mapping = to_level(0,1)

Out Matrix of shape (8, 93645)


In [18]:
top_20_resolved_to_level_1 = return_top_n(matrix, rows_mapping, col_mapping, axis=0,top_n=20)
top_20_resolved_to_level_1

,count
markeng_8,25054.0
gg_3,17785.0
vwgo_132,15521.0
gg_2,12713.0
gg_103,11943.0
fgo_115,9911.0
gg_20,8388.0
bgb_307,7879.0
zpo_563,7872.0
stpo_349,7764.0


In [19]:
matrix, rows_mapping, col_mapping = to_level(0, 2)

Out Matrix of shape (8, 268552)


In [20]:
top_20_resolved_to_level_2 = return_top_n(matrix, rows_mapping, col_mapping, axis=0,top_n=20)
top_20_resolved_to_level_2

,count
markeng_8_2,23665.500000
gg_3_1,15181.333333
vwgo_132_2,15144.666667
gg_103_1,10350.666667
fgo_115_2,9710.333333
gg_2_1,8348.000000
sgg_160_2,6999.666667
gg_19_4,6816.750000
gg_12_1,6589.333333
zpo_97_1,6182.000000


In [21]:
matrix, rows_mapping, col_mapping = to_level(0, 3)

Out Matrix of shape (8, 389518)


In [22]:
top_20_resolved_to_level_3 = return_top_n(
    matrix, rows_mapping, col_mapping, axis=0, top_n=20
)
top_20_resolved_to_level_3.join(levels_df)

,count,level
gg_3_1,15181.333333,2
markeng_8_2_1,12086.535714,3
gg_103_1,10350.666667,2
markeng_8_2_2,8788.535714,3
gg_2_1,8348.000000,2
vwgo_132_2_1,7454.555556,3
gg_19_4,6816.750000,2
gg_12_1,6589.333333,2
zpo_97_1,6182.000000,2
stpo_349_2,6034.800000,2


In [23]:
top_20_only_level_2

,count
item,
gg_3_1,14742
gg_103_1,10122
gg_2_1,7913
gg_19_4,6688
gg_12_1,6150
zpo_97_1,6100
stpo_349_2,6029
gg_20_3,5356
vwgo_154_2,5212


In [24]:
top_20_only_level_2.loc['gg_3_1','count']

np.int64(14742)

In [25]:
top_20_resolved_to_level_1.rank(ascending=False)

,count
markeng_8,1.0
gg_3,2.0
vwgo_132,3.0
gg_2,4.0
gg_103,5.0
fgo_115,6.0
gg_20,7.0
bgb_307,8.0
zpo_563,9.0
stpo_349,10.0


In [26]:
top_20_only_level_1

,count,level
item,,
bgb_242,4620,1
bgb_826,4089,1
stgb_64,2461,1
sgg_163,2432,1
sgg_193,2400,1
bgb_133,2388,1
bgb_134,2261,1
sgg_169,2245,1
zpo_561,2202,1


## 3. Comparison Tables

In [27]:
top_20_raw_table = top_20_raw.copy()
top_20_raw_table.index = get_well_written_sections(top_20_raw.index).values()
top_20_raw_table

,count,level
Art. 3 Abs. 1 GG,14742,2
§ 8 Abs. 2 Nr. 1 MarkenG,12053,3
Art. 103 Abs. 1 GG,10122,2
§ 8 Abs. 2 Nr. 2 MarkenG,8755,3
Art. 2 Abs. 1 GG,7913,2
§ 132 Abs. 2 Nr. 1 VwGO,6879,3
Art. 19 Abs. 4 GG,6688,2
Art. 12 Abs. 1 GG,6150,2
§ 97 Abs. 1 ZPO,6100,2
§ 349 Abs. 2 StPO,6029,2


In [28]:
pd.DataFrame().to_latex(
    columns=[
       rf"{i}" for i in  top_20_raw_table.index
    ]
)

'\\begin{tabular}{l}\n\\toprule\n\\midrule\n\\bottomrule\n\\end{tabular}\n'

In [29]:
top_20_raw_table

,count,level
Art. 3 Abs. 1 GG,14742,2
§ 8 Abs. 2 Nr. 1 MarkenG,12053,3
Art. 103 Abs. 1 GG,10122,2
§ 8 Abs. 2 Nr. 2 MarkenG,8755,3
Art. 2 Abs. 1 GG,7913,2
§ 132 Abs. 2 Nr. 1 VwGO,6879,3
Art. 19 Abs. 4 GG,6688,2
Art. 12 Abs. 1 GG,6150,2
§ 97 Abs. 1 ZPO,6100,2
§ 349 Abs. 2 StPO,6029,2


In [30]:
latex = (
    top_20_raw_table
    .reset_index(names="Section")  # move index → column
    .rename(columns={
        "Section": r"\textbf{Law}",
        "count": r"\textbf{Count}",
        "level": r"\textbf{Level}",
    })
    .to_latex(
        EXPORT_PATH / 'raw.tex'
        ,index=False, escape=False)
)
latex

In [31]:
def get_section_tree_id(sem_id: str):
    filtered_df = edge_list_df.query(f'target=="{sem_id}"').reset_index()
    if filtered_df.empty:
        return sem_id
    row = filtered_df.iloc[0]
    source, target = row["source"], row["target"]
    print(source)
    print(target)
    return sem_id if source == target else get_section_tree_id(source)


In [35]:

color_cycle = plt.get_cmap("tab20").__dict__["colors"]
colors = cycle(color_cycle)  # Add colors based on section tree
color_look_up = {}


def section_tree_color(sem_id, as_latex_dot: bool = True):
    section_tree_id = get_section_tree_id(sem_id=sem_id)
    if section_tree_id in color_look_up:
        c = color_look_up[section_tree_id]
        return get_dot_string(c) if as_latex_dot else c
    new_color = next(colors)
    color_look_up[section_tree_id] = new_color
    return get_dot_string(new_color) if as_latex_dot else new_color


In [36]:
def get_dot_string(color: tuple[float]) -> str:
    colors_string = f"{round(color[0], 4)},{round(color[1], 4)},{round(color[2], 4)}"
    latex_string = "\\textcolor[rgb]{color_string}{$\\bullet$}"
    return latex_string.replace("color_string", colors_string)

In [37]:
matrix, rows_mapping, col_mapping = to_level(0, 1)

Out Matrix of shape (8, 93645)


In [38]:
def get_parent_references(sem_id: str, recursive: bool = True, relative: bool = True):
    """gets number of references made to a"""
    refs = 0
    parent_lookup_df = edge_list_df.query(f'target=="{sem_id}"')
    if parent_lookup_df.empty:
        return refs
    else:
        parent = parent_lookup_df.iloc[0]["source"]
        if sem_id == parent:
            return 0
        refs_to_parent = occurrences_df.query(f"item=='{parent}'").__len__()

        if relative:
            num_of_children = edge_list_df.query(f'source=="{parent}"').__len__()
            refs += refs_to_parent / num_of_children
        if recursive:
            refs += get_parent_references(
                parent, recursive=recursive, relative=relative
            )
    return refs


In [39]:
def get_children_references(sem_id: str, recursive: bool = True, relative: bool = True):
    """gets number of references made to a"""
    refs = 0
    children_lookup_df = edge_list_df.query(f'source=="{sem_id}"')
    if children_lookup_df.empty:
        return refs
    else:
        children = children_lookup_df["target"]
        if sem_id in children.to_list():
            # print(f'{sem_id} is in {children}')
            return 0
        for c in children:
            # print(c)
            refs_to_child = occurrences_df.query(f"item=='{c}'").__len__()

            # if relative:
            #     num_of_children = edge_list_df.query(f'source=="{parent}"').__len__()
            #     refs += refs_to_parent / num_of_children
            refs += refs_to_child
            if recursive:
                refs += get_children_references(
                    c, recursive=recursive, relative=relative
                )
    return refs


assert get_children_references("bgb_242") == 0
assert get_children_references("markeng_8") != 0

In [40]:
def get_num_of_direct_citations(sem_id):
    return occurrences_df.query(f"item=='{sem_id}'").__len__()

In [41]:
def present_resolved_table(level: int = 2):
    matrix, rows_mapping, col_mapping = to_level(0, level)

    df = return_top_n(matrix, rows_mapping, col_mapping, axis=0, top_n=20).join(
        levels_df
    )
    df["is_from_higher_level"] = df["level"] < level
    df["is_from_higher_level"] = df["is_from_higher_level"].map(lambda x: "Yes" if x else "No")
    df["count_rounded"] = round(df["count"])

    # get references from higher and lower levels
    df["from_higher"] = df.index.map(get_parent_references)
    df["from_higher_share"] = round(df["from_higher"] / df["count"], 4) * 100

    df["from_lower"] = df.index.map(get_children_references)
    df["from_lower_share"] = round(df["from_lower"] / df["count"], 4) * 100

    df["direct_citations"] = df.index.map(get_num_of_direct_citations)
    df["direct_citations_share"] = round(df["direct_citations"] / df["count"], 4) * 100

    df["Section"] = df.index.map(get_well_written_section)
    df['color'] = df.index.map(section_tree_color)
    return df


df = present_resolved_table(level=2)
df

Out Matrix of shape (8, 268552)
markeng_8
markeng_8_2
gg_3
gg_3_1
vwgo_132
vwgo_132_2
gg_103
gg_103_1
fgo_115
fgo_115_2
gg_2
gg_2_1
sgg_160
sgg_160_2
gg_19
gg_19_4
gg_12
gg_12_1
zpo_97
zpo_97_1
stpo_349
stpo_349_2
gg_20
gg_20_3
vwgo_154
vwgo_154_2
bgb_823
bgb_823_2
gg_14
gg_14_1
bgb_307
bgb_307_1
bgb_242
bgb_242
zpo_563
zpo_563_1
gg_2
gg_2_2
vwgo_133
vwgo_133_3


,count,level,is_from_higher_level,count_rounded,from_higher,from_higher_share,from_lower,from_lower_share,direct_citations,direct_citations_share,Section,color
markeng_8_2,23665.500000,2,No,23666.0,124.500000,0.53,23196,98.02,345,1.46,§ 8 Abs. 2 MarkenG,"\textcolor[rgb]{0.1216,0.4667,0.7059}{$\bullet$}"
gg_3_1,15181.333333,2,No,15181.0,354.333333,2.33,0,0.00,14742,97.11,Art. 3 Abs. 1 GG,"\textcolor[rgb]{0.6824,0.7804,0.9098}{$\bullet$}"
vwgo_132_2,15144.666667,2,No,15145.0,149.666667,0.99,13418,88.60,1577,10.41,§ 132 Abs. 2 VwGO,"\textcolor[rgb]{1.0,0.498,0.0549}{$\bullet$}"
gg_103_1,10350.666667,2,No,10351.0,52.666667,0.51,0,0.00,10122,97.79,Art. 103 Abs. 1 GG,"\textcolor[rgb]{1.0,0.7333,0.4706}{$\bullet$}"
fgo_115_2,9710.333333,2,No,9710.0,77.333333,0.80,8599,88.56,1034,10.65,§ 115 Abs. 2 FGO,"\textcolor[rgb]{0.1725,0.6275,0.1725}{$\bullet$}"
gg_2_1,8348.000000,2,No,8348.0,118.000000,1.41,0,0.00,7913,94.79,Art. 2 Abs. 1 GG,"\textcolor[rgb]{0.5961,0.8745,0.5412}{$\bullet$}"
sgg_160_2,6999.666667,2,No,7000.0,11.666667,0.17,6486,92.66,502,7.17,§ 160 Abs. 2 SGG,"\textcolor[rgb]{0.8392,0.1529,0.1569}{$\bullet$}"
gg_19_4,6816.750000,2,No,6817.0,7.750000,0.11,0,0.00,6688,98.11,Art. 19 Abs. 4 GG,"\textcolor[rgb]{1.0,0.5961,0.5882}{$\bullet$}"
gg_12_1,6589.333333,2,No,6589.0,354.000000,5.37,0,0.00,6150,93.33,Art. 12 Abs. 1 GG,"\textcolor[rgb]{0.5804,0.4039,0.7412}{$\bullet$}"
zpo_97_1,6182.000000,2,No,6182.0,82.000000,1.33,0,0.00,6100,98.67,§ 97 Abs. 1 ZPO,"\textcolor[rgb]{0.7725,0.6902,0.8353}{$\bullet$}"


In [42]:
def export_tabular(df, path=None, t_at_start: bool = False):
    df = df.copy()

    # Move index into column
    df = df.reset_index()
    df = df.rename(columns={"index": "Section"})  # or whatever your index name is

    # Rounding
    df["count_rounded"] = df["count_rounded"].round(0).astype(int)

    share_cols = ["from_higher_share", "direct_citations_share", "from_lower_share"]
    df[share_cols] = df[share_cols].round(2)

    data_cols = [
            "Section",
            "count_rounded",
            "from_higher_share",
            "direct_citations_share",
            "from_lower_share",
            "is_from_higher_level",
        
        ]
    base_cols = [
        ("", r"\textbf{Section}"),
        ("", r"\textbf{Count}"),
        (r"\textbf{Share in \%}", r"\textbf{H}"),
        (r"\textbf{Share in \%}", r"\textbf{D}"),
        (r"\textbf{Share in \%}", r"\textbf{L}"),
        ("", r"\textbf{P}"),
    ]

    t_col = ("", r"\textbf{T}")

    if not t_at_start:
        columns = base_cols + [t_col]
        data_cols = data_cols + ['color']
    else:
        columns = [t_col] + base_cols
        data_cols = ['color'] + data_cols

    # Select columns
    df_out = df[
        data_cols
    ].copy()
   
    share_cols = [
        "from_higher_share",
        "direct_citations_share",
        "from_lower_share",
    ]

    df_out[share_cols] = df_out[share_cols].apply(pd.to_numeric, errors="coerce")

    df_out.columns = pd.MultiIndex.from_tuples(columns)

    # Format
    styler = df_out.style.format(
        {
            (r"\textbf{Share in \%}", r"\textbf{H}"): "{:.2f}",
            (r"\textbf{Share in \%}", r"\textbf{D}"): "{:.2f}",
            (r"\textbf{Share in \%}", r"\textbf{L}"): "{:.2f}",
        }
    ).hide(axis="index")

    latex = styler.to_latex(
        hrules=True,
        multicol_align="c",
    )

    # Optional alignment tweak
    latex = latex.replace("\\begin{tabular}{llllll}", "\\begin{tabular}{lrrrrl}")

    if path:
        with open(path, "w") as f:
            f.write(latex)

    return latex

In [43]:
cleaned = present_resolved_table(level=2)[
    [
        "count_rounded",
        "Section",
        "from_higher_share",
        "direct_citations_share",
        "from_lower_share",
        "is_from_higher_level",
        'color'
    ]
].set_index("Section")
print(export_tabular(cleaned, path=EXPORT_PATH / "level_2.tex", t_at_start=True))

Out Matrix of shape (8, 268552)
markeng_8
markeng_8_2
gg_3
gg_3_1
vwgo_132
vwgo_132_2
gg_103
gg_103_1
fgo_115
fgo_115_2
gg_2
gg_2_1
sgg_160
sgg_160_2
gg_19
gg_19_4
gg_12
gg_12_1
zpo_97
zpo_97_1
stpo_349
stpo_349_2
gg_20
gg_20_3
vwgo_154
vwgo_154_2
bgb_823
bgb_823_2
gg_14
gg_14_1
bgb_307
bgb_307_1
bgb_242
bgb_242
zpo_563
zpo_563_1
gg_2
gg_2_2
vwgo_133
vwgo_133_3
\begin{tabular}{llrrrrl}
\toprule
\multicolumn{3}{c}{} & \multicolumn{3}{c}{\textbf{Share in \%}} &  \\
\textbf{T} & \textbf{Section} & \textbf{Count} & \textbf{H} & \textbf{D} & \textbf{L} & \textbf{P} \\
\midrule
\textcolor[rgb]{0.1216,0.4667,0.7059}{$\bullet$} & § 8 Abs. 2 MarkenG & 23666 & 0.53 & 1.46 & 98.02 & No \\
\textcolor[rgb]{0.6824,0.7804,0.9098}{$\bullet$} & Art. 3 Abs. 1 GG & 15181 & 2.33 & 97.11 & 0.00 & No \\
\textcolor[rgb]{1.0,0.498,0.0549}{$\bullet$} & § 132 Abs. 2 VwGO & 15145 & 0.99 & 10.41 & 88.60 & No \\
\textcolor[rgb]{1.0,0.7333,0.4706}{$\bullet$} & Art. 103 Abs. 1 GG & 10351 & 0.51 & 97.79 & 0.00 & No \

In [44]:
cleaned = present_resolved_table(level=1)[
    [
        "count_rounded",
        "Section",
        "from_higher_share",
        "direct_citations_share",
        "from_lower_share",
        "is_from_higher_level",
        "color",
    ]
].set_index("Section")
print(export_tabular(cleaned, path=EXPORT_PATH / "level_1.tex", t_at_start=False))

Out Matrix of shape (8, 93645)
\begin{tabular}{lrrrrll}
\toprule
\multicolumn{2}{c}{} & \multicolumn{3}{c}{\textbf{Share in \%}} & \multicolumn{2}{c}{} \\
\textbf{Section} & \textbf{Count} & \textbf{H} & \textbf{D} & \textbf{L} & \textbf{P} & \textbf{T} \\
\midrule
§ 8 MarkenG & 25054 & 0.00 & 1.99 & 98.01 & No & \textcolor[rgb]{0.1216,0.4667,0.7059}{$\bullet$} \\
Art. 3 GG & 17785 & 0.00 & 5.98 & 93.01 & No & \textcolor[rgb]{0.6824,0.7804,0.9098}{$\bullet$} \\
§ 132 VwGO & 15521 & 0.00 & 2.89 & 97.11 & No & \textcolor[rgb]{1.0,0.498,0.0549}{$\bullet$} \\
Art. 2 GG & 12713 & 0.00 & 1.86 & 93.55 & No & \textcolor[rgb]{0.5961,0.8745,0.5412}{$\bullet$} \\
Art. 103 GG & 11943 & 0.00 & 1.32 & 96.82 & No & \textcolor[rgb]{1.0,0.7333,0.4706}{$\bullet$} \\
§ 115 FGO & 9911 & 0.00 & 2.34 & 97.66 & No & \textcolor[rgb]{0.1725,0.6275,0.1725}{$\bullet$} \\
Art. 20 GG & 8388 & 0.00 & 4.86 & 93.18 & No & \textcolor[rgb]{0.7686,0.6118,0.5804}{$\bullet$} \\
§ 307 BGB & 7879 & 0.00 & 16.45 & 83.55 & No